### 시나리오
- AI관련 교육상담을 하는 챗봇을 구성한다
- 최근 정부에서 '에이아이캠퍼스'라는 교육사업을 추친 중
- 해당 사업에 대한 공고문(pdf)
- 해당 사업에 맞는 광고 홍보 페이지
  - https://smhrd.or.kr/course/aic/

### 학습내용
- Mulit-Source RAG 실습 (PDF와 웹페이지 retriver 정보 결합)
- Query Transformation 실습
  - ex) '에이아이캠퍼스 교육과정 어때?' -> 사용자는 보통 질문을 축약해서 하는 편
  - LLM에 변경요청 -> '에이아이캠퍼스의 사업목표, 교육과정종류, 시수, 커리큘럼 등을 알려줘'

### 환경설정

In [ ]:
# open AI API key 등록
import os
OPENAI_API_KEY=""
# 현재 노트북 커널에 환경변수 등록
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY


# LangSmith API key 등록
LANGSMITH_TRACING="true" # 추적 할지 말지
LANGSMITH_ENDPOINT="https://api.smith.langchain.com"
LANGSMITH_API_KEY=""
LANGSMITH_PROJECT="Langchain0422"
# 현재 노트북 커널에 환경변수 등록
os.environ['LANGSMITH_TRACING'] = LANGSMITH_TRACING
os.environ['LANGSMITH_ENDPOINT'] = LANGSMITH_ENDPOINT
os.environ['LANGSMITH_API_KEY'] = LANGSMITH_API_KEY
os.environ['LANGSMITH_PROJECT'] = LANGSMITH_PROJECT

In [2]:
!pip install langchain langchain-openai langchain-community faiss-cpu pypdf chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 79.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180

### 1. Query Transformation

In [108]:
# RAG 관련 도구
from langchain_community.document_loaders import PyPDFLoader # PDF 로더
from langchain_text_splitters import RecursiveCharacterTextSplitter # 재귀적 텍스트 분할 도구
from langchain_openai import OpenAIEmbeddings # 임베딩 도구
from langchain_community.vectorstores import FAISS # 벡터데이터베이스 도구
from langchain_core.documents import Document # 문서 도구
# Chain 구성 도구
from langchain.chat_models import init_chat_model # LLM 생성 도구
from langchain_core.prompts import ChatPromptTemplate # 프롬프트 템플릿
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser # 풀력파서 도구
from langchain_core.runnables import RunnablePassthrough, RunnableBranch, RunnableParallel, RunnableLambda # 구조생성 Runnable 도구
from langchain_core.prompts import PromptTemplate

In [34]:
QT_template = ChatPromptTemplate.from_template('''
  너는 사용자의 query를 retriver가 더 잘 검색하도록 최적화하는 역할이야.

  - 수행내용
    1. 아래 query를 기반으로 사용자의 질문 의도를 추출
    2. 사용자 질문을 깔끔하고 정돈된 질문으로 변환
    3. 사용자의 질문을 다양한 관점으로 확장해서 3가지 정도로 늘려줘

  - query: {query}

  - 출력형식
    아래 key값을 이용해서 파싱가능한 JSON 구조로 만들어줘

    key:
      - user_inient
      - normalize_query
      - expanded_query
''')

In [35]:
# 체인구성하기
llm_4o_mini = init_chat_model('openai:gpt-4o-mini', max_tokens=1024)
jsonParser = JsonOutputParser()
QT_chain = QT_template | llm_4o_mini | jsonParser

In [36]:
# q1
QT_chain.invoke("에이아이캠퍼스 교육과정 어때")

{'user_intent': '에이아이캠퍼스의 교육과정에 대한 평가 및 정보 조회',
 'normalize_query': '에이아이캠퍼스의 교육과정은 어떤가요?',
 'expanded_query': ['에이아이캠퍼스 교육과정의 강점과 약점은 무엇인가요?',
  '에이아이캠퍼스에서 제공하는 교육과정의 특징은 어떤 것이 있나요?',
  '다른 교육기관과 비교했을 때 에이아이캠퍼스의 교육과정은 어떤 점이 특별한가요?']}

In [37]:
QT_chain.invoke("에이아이캠퍼스 과정 빡세?")

{'user_intent': 'AI 캠퍼스 과정의 난이도에 대한 질문',
 'normalize_query': 'AI 캠퍼스 과정은 얼마나 어려운가요?',
 'expanded_query': ['AI 캠퍼스 과정의 난이도는 어떤가요?',
  'AI 캠퍼스 과정을 공부하기 위해 어떤 준비가 필요할까요?',
  'AI 캠퍼스 과정이 수강생들에게 힘든 경험으로 평가되나요?']}

In [38]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [39]:
# 파일로더 생성
loader = PyPDFLoader("/content/drive/MyDrive/Colab Notebooks/딥러닝/LLM 활용/data/260107 K-디지털 트레이닝 AI 캠퍼스 운영(인적자원개발과).pdf")
# 데이터로딩
documents = loader.load()

In [40]:
len(documents)

4

In [41]:
type(documents)

list

In [42]:
type(documents[0])

langchain_core.documents.base.Document

In [48]:
pdf_splitter = RecursiveCharacterTextSplitter(
    separators= ['\n\n', '\n', ' ',''],
    chunk_size = 500, # 청크 사이즈
    chunk_overlap = 100 # 청크 오버랩 사이즈
)

In [49]:
pdf_chunk = pdf_splitter.split_documents(documents)

In [51]:
len(pdf_chunk)

12

In [52]:
# 임베딩 및 인덱싱
vec_db = FAISS.from_documents(
    documents=pdf_chunk, # 임베딩할 벡터
    embedding=OpenAIEmbeddings() # 임베딩 도구 연결
)

In [53]:
# 사용자 질문과 유사한 청크 2개를 찾는 검색기 생성
pdf_retriever = vec_db.as_retriever(search_kwargs = {'k' : 2})

In [54]:
# 테스트
pdf_retriever.invoke('AI캠퍼스 과정은 얼마나 어려운가요?')

[Document(id='4cb51532-17bc-4588-86b8-291f05ebe527', metadata={'producer': 'Hancom PDF 1.3.0.550', 'creator': 'Hwp 2022 12.0.0.4204', 'creationdate': '2026-01-06T14:12:03+09:00', 'author': 'Moel', 'moddate': '2026-01-06T14:12:03+09:00', 'pdfversion': '1.4', 'source': '/content/drive/MyDrive/Colab Notebooks/딥러닝/LLM 활용/data/260107 K-디지털 트레이닝 AI 캠퍼스 운영(인적자원개발과).pdf', 'total_pages': 4, 'page': 3, 'page_label': '4'}, page_content='붙임 AI 캠퍼스 양성 목표 직종(직무) 직군분류 (예시 직종/직무) 직무정의① AI 엔지니어(AI Engineer)"어 떻 게  구 현 하 는 가?" 연구자가 밝혀낸 원리를 바탕으로, 실 제  사 용 자 가  쓸 수 있는 안정적인 서비스를 구축하고 운영'),
 Document(id='cc7f421d-14d2-416d-8701-225384d6a4ad', metadata={'producer': 'Hancom PDF 1.3.0.550', 'creator': 'Hwp 2022 12.0.0.4204', 'creationdate': '2026-01-06T14:12:03+09:00', 'author': 'Moel', 'moddate': '2026-01-06T14:12:03+09:00', 'pdfversion': '1.4', 'source': '/content/drive/MyDrive/Colab Notebooks/딥러닝/LLM 활용/data/260107 K-디지털 트레이닝 AI 캠퍼스 운영(인적자ᄋ

### 3. 웹페이지기반 retriever 구성하기
- 기본 WebBaseLoader는 정적페이지는 잘 불러옴
- 동적페이지를 불러오려면 외부도구나 selenium을 활용하는것이 표과적이다.

In [55]:
# 앞쪽에서 selenium을 통해 웹문서의 text를 추출했다는 가정하에 진행
text1 = '''
  6개월 과정960시간오프라인
파이썬과 AI수학
- 파이썬 분석/시각화, 선형대수, 확률, 통계

컴퓨터 비전
- 영상처리, 객체탐지/인식, 동작인식, 얼굴인식, 영역분할

머신러닝/딥러닝
- 지도/비지도학습, 평가지표, 신경망, CNN, RNN, Transformer, LMM/VLM/VLA

로봇 운동학
- 3차원공간 이해, 좌표계 변환, 자유도, FK/IK 해석, 로봇 움직임 제어, PID

ROS2과 SLAM
- ROS2, 거리 및 IMU 센서, 점군 데이터 처리, SLAM, Navigation

가상 로봇 설계
- URDF/SDF기반 설계, 가상센서, 가상 로봇 제어

가상 로봇 제어
- 맵핑, 장애물 회피, 복구행동, 경로계획, 행동결정, 로봇팔, 다중로봇, 자율주행

현실 로봇 제어
- ROS brigde, 카메라/Lidar 연동, 모터 제어, 자율주행

배포 서비스 구현
- Streamlit/FastAPI

AI 서비스 설계
- AI 웹 서비스 기술 (HTML/CSS, JS, Node.js. React.js)
'''

In [56]:
doc1 = Document(
    page_content = text1,
    metadata = {'source' : 'https://smhrd.or.kr/course/aic_network/'}
)

In [57]:
doc1

Document(metadata={'source': 'https://smhrd.or.kr/course/aic_network/'}, page_content='\n  6개월 과정960시간오프라인\n파이썬과 AI수학\n- 파이썬 분석/시각화, 선형대수, 확률, 통계\n\n컴퓨터 비전\n- 영상처리, 객체탐지/인식, 동작인식, 얼굴인식, 영역분할\n\n머신러닝/딥러닝\n- 지도/비지도학습, 평가지표, 신경망, CNN, RNN, Transformer, LMM/VLM/VLA\n\n로봇 운동학\n- 3차원공간 이해, 좌표계 변환, 자유도, FK/IK 해석, 로봇 움직임 제어, PID\n\nROS2과 SLAM\n- ROS2, 거리 및 IMU 센서, 점군 데이터 처리, SLAM, Navigation\n\n가상 로봇 설계\n- URDF/SDF기반 설계, 가상센서, 가상 로봇 제어\n\n가상 로봇 제어\n- 맵핑, 장애물 회피, 복구행동, 경로계획, 행동결정, 로봇팔, 다중로봇, 자율주행\n\n현실 로봇 제어\n- ROS brigde, 카메라/Lidar 연동, 모터 제어, 자율주행\n\n배포 서비스 구현\n- Streamlit/FastAPI\n\nAI 서비스 설계\n- AI 웹 서비스 기술 (HTML/CSS, JS, Node.js. React.js)\n')

In [58]:
text2 = '''
6개월 과정960시간오프라인
파이썬과 AI수학
- 파이썬 분석/시각화, 선형대수, 확률, 통계

데이터 수집/전처리
- 크롤링, 데이터 구조 (JSON, XML, CSV 등)

머신러닝/딥러닝
- 지도/비지도학습, 평가지표, 신경망, CNN, RNN, Transformer, LMM/VLM/VLA

컴퓨터 비전
- 영상처리, 객체탐지/인식, 동작인식, 얼굴인식, 영역분할

텍스트마이닝
- 텍스트 전처리, 임베딩, 기본 모델 실습, Transformer, 모델 활용 실습

음성데이터 / 시계열 데이터
- 텍스트 전처리, 임베딩, 기본 모델 실습, Transformer, 모델 활용 실습

LangChain
- 프롬프트, 템플릿, 체인 생성/연결, 메모리, RAG, 고급 RAG

AI Agent
- LangServe, LangSmith, Ollama, LangGraph

Vision Agent
- 종류, 구현실습, LangGraph 기반 구현

배포 서비스
- Streamlit / FastAPI / n8n

AI 서비스 설계
- AI 웹 서비스 기술(HTML/CSS, JS, Node.js, React.js)

'''

In [59]:
doc2 = Document(
    page_content = text2,
    metadata = {'source' : 'https://smhrd.or.kr/course/aic_network/'}
)

In [60]:
text3 = '''
6개월 과정960시간오프라인
헬스케어 AI와 파이썬
- 헬스케어 산업/기술/현황, 파이썬 분석/시각화, 선형대수, 확률, 통계

Java
- 정보공학, 환경설정, 연산자, 제어문, 배열, OOP, 예외처리, I/O, 쓰레드

JSP/Spring
- HTML, CSS, JS, JSP, Spring, Spring AI

데이터베이스
- 데이터베이스 개론, RDBMS실습, NoSQL

데이터 수집/전처리
- 헬스케어 데이터 크롤링, 데이터 구조 (JSON, XML, CSV 등)

머신러닝/딥러닝
- 지도/비지도학습, 평가지표, 신경망, CNN, RNN, Transformer, LMM/VLM/VLA

컴퓨터 비전
- 영상처리, 객체탐지/인식, 동작인식, 얼굴인식, 영역분할

텍스트마이닝
- 텍스트 전처리, 임베딩, 기본 모델 실습, Transformer, 모델 활용 실습

LangChain
- 프롬프트, 템플릿, 체인 생성 /연결, 메모리, RAG, 고급 RAG

n8n
- 노드, 워크플로우 설계, 인증, AI 자동화, MCP
'''

In [61]:
doc3 = Document(
    page_content = text3,
    metadata = {'source' : 'https://smhrd.or.kr/course/aic_network/'}
)

In [62]:
text4 = '''
6개월 과정960시간오프라인
지역산업 AI 비즈니스 이해
- 지역특화산업/현황, AI 서비스 기획, 비즈니스브리프, 고객 여정맵, 시장조사, KPI

고객경험 도메인 데이터 설계
- 경험데이터 분석, DCX개념에 따른 고객맞춤형 서비스 모델 이해, 아이디어 토론 및 공유

X+AI와 파이썬
- 지역도메인AI현황 분석, 파이썬 분석/시각화, 선형대수, 확률, 통계

데이터베이스
- 데이터베이스 개론, RDBMS실습, NoSQL

통계적분석기반 도메인 데이터 이해
- 실데이터 분석, 시나리오 정의, 인사이트 도출 및 솔루션 설계

고객경험 도메인 데이터 수집
- HTML, CSS, Web Crawling, 고객경험 데이터 수집/분석/ 서비스 기획 실습. 수집된 데이터 활용 방안 수립

디지털 맥락 파악 및 고객 감성분석
- 머신러닝, 텍스트마이닝, 시각화, 감정분석, Word2Vec, 키워드 추출

X+AI 비지니스 가치 설계
- 페르소나정의, ActionMap 작성, 기회영역 분석, 고객경험 디자인수행, AI기술, BMC, AI 윤리

AI활용 고객경험 분석
- 딥러닝, 컴퓨터 비전, 텍스트마이닝, 챗봇, 음성처리

도메인데이터 분석기반 MVP 설계
- 프로젝트관리, KPI/OKR, 사용자 피드백, UX, 서비스 블루프린트, PoC, 노코드 AI 도구, 프로토타입 제작

'''

In [64]:
doc4 = Document(
    page_content = text4,
    metadata = {'source' : 'https://smhrd.or.kr/course/aic_network/'}
)

In [65]:
# 4개의 문서를 하나의 리스트에 추가하여 웹 청크로 구성
web_chunk = [doc1, doc2, doc3, doc4]

In [66]:
web_chunk

[Document(metadata={'source': 'https://smhrd.or.kr/course/aic_network/'}, page_content='\n  6개월 과정960시간오프라인\n파이썬과 AI수학\n- 파이썬 분석/시각화, 선형대수, 확률, 통계\n\n컴퓨터 비전\n- 영상처리, 객체탐지/인식, 동작인식, 얼굴인식, 영역분할\n\n머신러닝/딥러닝\n- 지도/비지도학습, 평가지표, 신경망, CNN, RNN, Transformer, LMM/VLM/VLA\n\n로봇 운동학\n- 3차원공간 이해, 좌표계 변환, 자유도, FK/IK 해석, 로봇 움직임 제어, PID\n\nROS2과 SLAM\n- ROS2, 거리 및 IMU 센서, 점군 데이터 처리, SLAM, Navigation\n\n가상 로봇 설계\n- URDF/SDF기반 설계, 가상센서, 가상 로봇 제어\n\n가상 로봇 제어\n- 맵핑, 장애물 회피, 복구행동, 경로계획, 행동결정, 로봇팔, 다중로봇, 자율주행\n\n현실 로봇 제어\n- ROS brigde, 카메라/Lidar 연동, 모터 제어, 자율주행\n\n배포 서비스 구현\n- Streamlit/FastAPI\n\nAI 서비스 설계\n- AI 웹 서비스 기술 (HTML/CSS, JS, Node.js. React.js)\n'),
 Document(metadata={'source': 'https://smhrd.or.kr/course/aic_network/'}, page_content='\n6개월 과정960시간오프라인\n파이썬과 AI수학\n- 파이썬 분석/시각화, 선형대수, 확률, 통계\n\n데이터 수집/전처리\n- 크롤링, 데이터 구조 (JSON, XML, CSV 등)\n\n머신러닝/딥러닝\n- 지도/비지도학습, 평가지표, 신경망, CNN, RNN, Transformer, LMM/VLM/VLA\n\n컴퓨터 비전\n- 영상처리, 객체탐지/인식, 동작인식, 얼굴인식, 영역분할\n\n텍스트마이닝\n- 텍스트 전처리, 임베딩, 기본 모델 실습, Transf

In [67]:
# 임베딩 및 인덱싱
vec_db2 = FAISS.from_documents(
    documents=web_chunk, # 임베딩할 벡터
    embedding=OpenAIEmbeddings() # 임베딩 도구 연결
)

In [96]:
# 사용자 질문과 유사한 청크 2개를 찾는 검색기 생성
web_retriever = vec_db2.as_retriever(search_kwargs = {'k' : 1})

### 4. 교육과정 챗봇 구성하기
- 사용자의 질문에 커리큘럼 내용이 있으면 web_retriever가 수행되도록 branch 구성

In [97]:
extract_info = ChatPromptTemplate.from_template('''
  아래 query 데이터에서 사용자가 과정에 대한 질문을 하는지 파악하고
  과정 커리큘럼에 대한 질문이라면 특정 과정을 문의하는지 파악해서 알려줘

  출력형식은 아래와 같이 JSON 형태로 만들어줘. JSON 파싱이 가능하도록 꼭 포멧 시켜줘

  과정문의 : True, 과정타입 : 헬스케어

  만약 과정타입이 정보에 없다면 없음이라고 표기해

  query : {query}
''')

In [98]:
# 체인구성
test_chain = {'query' : QT_chain} | extract_info | llm_4o_mini | JsonOutputParser()

In [99]:
test_chain.invoke('에이아이캠퍼스의 교육과정에는 어떤 과목들이 포함되어 있나요?')

{'과정문의': True, '과정타입': '없음'}

In [100]:
test_chain.invoke('헬스케어 교육듣고 싶어요')

{'과정문의': True, '과정타입': '헬스케어'}

In [101]:
test_chain.invoke('에이아이캠퍼스 사업이 뭔가요?')

{'과정문의': False, '과정타입': '없음'}

#### 구조변경하기

In [102]:
extract_chain = extract_info | llm_4o_mini | JsonOutputParser()

In [103]:
test_chain2 = {'query' : QT_chain} | RunnableParallel(
    info = extract_chain,
    QT_result = RunnablePassthrough()
)

In [104]:
test_chain2.invoke('에이아이캠퍼스 사업이 뭐야?')

{'info': {'과정문의': False, '과정타입': '없음'},
 'QT_result': {'query': {'user_inient': '에이아이캠퍼스의 사업 내용에 대한 정보 탐색',
   'normalize_query': '에이아이캠퍼스의 사업이 무엇인지 알려주세요.',
   'expanded_query': ['에이아이캠퍼스는 어떤 종류의 사업을 운영하고 있나요?',
    '에이아이캠퍼스의 주요 사업 영역과 서비스는 무엇인가요?',
    '에이아이캠퍼스의 비즈니스 모델에 대해 설명해 주세요.']}}}

In [109]:
# 브렌치 추가하기
web_chain = RunnableParallel(
    context = RunnableLambda(lambda x : x['QT_result']) | RunnableLambda( lambda x : PromptTemplate.from_template('{query}').invoke(x).text) |web_retriever,
    question = RunnableLambda(lambda x : x['QT_result']['query'])
)
pdf_chain = RunnableParallel(
    context = RunnableLambda(lambda x : x['QT_result'])| RunnableLambda( lambda x : PromptTemplate.from_template('{query}').invoke(x).text) | pdf_retriever,
    question = RunnableLambda(lambda x : x['QT_result']['query'])
)

# 챗봇용 템플릿 생성
chatbot_template = ChatPromptTemplate.from_messages([
    ('system','너는 교육과정 응답을 하는 챗봇이야. 아래 context 내용과 question을 기반으로 답변해줘'),
    ('system', 'context : {context}'),
    ('human', 'question : {question}')
])

final_chain = test_chain2 | RunnableBranch(
                      (lambda x : x['info']['과정문의'] == True, lambda x : web_chain),
                      lambda x : pdf_chain
              ) | chatbot_template | llm_4o_mini | StrOutputParser()

In [110]:
final_chain.invoke('에이아이캠퍼스의 교육과정에는 어떤 과정들이 포함되어 있나요?')

'에이아이캠퍼스의 교육과정은 다음과 같은 다양한 과정으로 구성되어 있습니다:\n\n1. **파이썬과 AI수학**\n   - 파이썬 분석/시각화\n   - 선형대수, 확률, 통계\n\n2. **데이터 수집/전처리**\n   - 크롤링, 데이터 구조 (JSON, XML, CSV 등)\n\n3. **머신러닝/딥러닝**\n   - 지도/비지도학습, 평가지표, 신경망, CNN, RNN, Transformer, LMM/VLM/VLA\n\n4. **컴퓨터 비전**\n   - 영상처리, 객체탐지/인식, 동작인식, 얼굴인식, 영역분할\n\n5. **텍스트 마이닝**\n   - 텍스트 전처리, 임베딩, 기본 모델 실습, Transformer, 모델 활용 실습\n\n6. **음성 데이터 / 시계열 데이터**\n   - 텍스트 전처리, 임베딩, 기본 모델 실습, Transformer, 모델 활용 실습\n\n7. **LangChain**\n   - 프롬프트, 템플릿, 체인 생성/연결, 메모리, RAG, 고급 RAG\n\n8. **AI Agent**\n   - LangServe, LangSmith, Ollama, LangGraph\n\n9. **Vision Agent**\n   - 종류, 구현실습, LangGraph 기반 구현\n\n10. **배포 서비스**\n    - Streamlit / FastAPI / n8n\n\n11. **AI 서비스 설계**\n    - AI 웹 서비스 기술(HTML/CSS, JS, Node.js, React.js)\n\n이와 같이 다양한 주제를 포함한 커리큘럼을 통해 AI와 관련된 종합적인 교육을 제공합니다.'

### Colab에서 외부로 Langchain 연결하기

### 학습목표
- MVP를 구성할때 colab 환경에서 langchain을 구성하고 외부에서 호출 가능하게 만들 수 있다
- 실제 프로덕션 단계에서는 사용이 불가능, MVP 시연용으로 활용
- 또 GPU 환경으로 모델을 배포하는 경우에도 응용가능

### 패키지 설치
- langchain, langchain-openai : 랭체인 구성을 위한 패키지
- langserve : 랭체인으로 구성한 체인을 REST API로 감싸는 패키지
- fastapi, uvicorn : 파이썬 서버를 구성하기 위한 도구
- pyngrok : 터널링을 이용해 외부통신이 가능하게 해주는 도구

In [111]:
!pip install langserve fastapi uvicorn pyngrok

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.4 MB/s eta 0:00:00


### FastAPI + LangServer붙이기

In [112]:
from fastapi import FastAPI
from langserve import add_routes

In [114]:
app = FastAPI() # 서버생성
add_routes(app, # 라우팅을 추가할 서버
           final_chain, # 라우팅에 연결할 chain
           path = '/query' # 해당 체인을 접속할 URL path
           )

### Colab에서 서버 실행

In [116]:
import nest_asyncio
import uvicorn
from threading import Thread

In [124]:
nest_asyncio.apply() # 노트북 커널과 별개로 수행이 가능하도록 셀 셋팅

def run():
  uvicorn.run(app, # 구동할 서버앱
              host='0.0.0.0', # 노트북이 돌고있는 pc의 ip
              port=8000 # 통신이 가능한 포트(출입구)
              )

thread = Thread(target=run) # 쓰레드 생성
thread.start() # 별도의 쓰레드 구동

INFO:     Started server process [37329]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


#### colab 인스턴스 내부에서 호출하기

In [126]:
!curl -X POST "http://0.0.0.0:8000/query/invoke" \
-H "Content-Type: application/json" \
-d '{"input" : {"query" : "에이아이캠퍼스 사업이 뭐야?"}}'

INFO:     127.0.0.1:57614 - "POST /query/invoke HTTP/1.1" 200 OK
{"output":"에이아이캠퍼스는 주로 AI 관련 직무 교육 및 인력을 양성하는 것을 목표로 하는 사업을 수행합니다. 특히, AI 엔지니어와 같은 다양한 직종을 위한 교육 프로그램을 운영하여, 실제 사용자들이 안정적으로 사용할 수 있는 AI 서비스를 구축하고 운영할 수 있는 인재를 양성하는 데 중점을 두고 있습니다.  \n\n또한, 에이아이캠퍼스는 직업능력심사평가원과 협력하여 심사 및 평가 계획을 수립하고, 이를 통해 AI 관련 인력의 양성과 평가를 지원하는 프로그램을 제공할 예정입니다. \n\n이 외에도 에이아이캠퍼스는 다양한 분야에서 AI 활용과 관련된 교육 및 자격증 프로그램을 통해, 관련 산업의 인력 수요에 맞춘 맞춤형 교육 서비스를 제공하고 있습니다. 비즈니스 모델에 대한 구체적인 정보나 시장 점유율은 아직 명시된 바가 없어 확인이 필요한 부분입니다.","metadata":{"run_id":"a3c31460-eae0-4664-bb4b-4cfae8ec2df3","feedback_tokens":[]}}

### ngrok 터널 열기
- 터널링을 통해 외부통신이 가능하도록 지원하는 서비스

In [ ]:
# 토큰등록
from pyngrok import ngrok
ngrok.set_auth_token("")

In [128]:
public_url = ngrok.connect(8000) # 현재 PC에서 서버가 돌고있는 포트 연결

In [129]:
# 무료플랜이어서 URL은 변동 됨
public_url

<NgrokTunnel: "https://animator-unsavory-tipoff.ngrok-free.dev" -> "http://localhost:8000">